# `17 — Rabin–Karp algorithm`

Goal:
Find all occurrences of pattern $t$ in text $s$.

Naive:
For each shift $i$, compare characters $\Rightarrow O(|s| \cdot |t|)$

Rabin–Karp idea:
- compare **hashes** of substrings instead of comparing whole strings each time
- compute prefix polynomial hashes for $s$ in $O(|s|)$
- compute hash for $t$ in $O(|t|)$
- each shift comparison is $O(1)$ for hash equality
- if hashes match, do a **final explicit string check** to avoid false positives

Complexity:
$O(|s| + |t| + |t| \cdot (\text{matches} + \text{collisions}))$

Collision probability is about $\frac{1}{p}$ if $p$ is a large prime.


In [1]:
# Rabin–Karp with full visualization of rolling hash checks.
from typing import List, Tuple


def char_to_int(c: str) -> int:
    # Simple mapping (works for lowercase a..z)
    return ord(c) - ord("a") + 1

def build_poly_prefix_hash(s: str, *, a: int, p: int, verbose: bool = True) -> Tuple[List[int], List[int]]:
    """
    h[i] = hash(s[:i]) = sum_{j=0..i-1} s[j]*a^j mod p
    pow_a[i] = a^i mod p
    """
    n: int = len(s)
    h: List[int] = [0] * (n + 1)
    pow_a: List[int] = [1] * (n + 1)

    if verbose:
        print("-" * 80)
        print("Building polynomial prefix hash")
        print(f"s={s!r}, base a={a}, mod p={p}")
        print("-" * 80)

    for i in range(n):
        h[i + 1] = (h[i] + char_to_int(s[i]) * pow_a[i]) % p
        pow_a[i + 1] = (pow_a[i] * a) % p
        if verbose:
            print(f"i={i:>2} char={s[i]!r} val={char_to_int(s[i])} "
                  f"-> h[{i+1}]={h[i+1]}, a^{i+1}={pow_a[i+1]}")

    return h, pow_a

def rabin_karp_trace(s: str, *, t: str, a: int, p: int, verbose: bool = True) -> List[int]:
    n: int = len(s)
    m: int = len(t)

    if m == 0:
        return list(range(n + 1))
    if m > n:
        return []

    # Prefix hashes for s
    h_s, pow_a = build_poly_prefix_hash(s, a=a, p=p, verbose=False)

    # hash for t (same polynomial form)
    h_t: int = 0
    for i, ch in enumerate(t):
        h_t = (h_t + char_to_int(ch) * pow_a[i]) % p

    if verbose:
        print("-" * 90)
        print("Rabin–Karp trace")
        print(f"s={s!r}")
        print(f"t={t!r}")
        print(f"base a={a}, mod p={p}")
        print(f"hash(t) (h_t) = {h_t}")
        print("-" * 90)

    hits: List[int] = []

    for i in range(0, n - m + 1):
        # Compare: h_t * a^i == (h_s[i+m] - h_s[i])  (mod p)
        left = (h_t * pow_a[i]) % p
        right = (h_s[i + m] - h_s[i]) % p

        if verbose:
            print(f"\nshift i={i}")
            print("s:", s)
            print("t:", " " * i + t)
            print(f"compare hashes: (h_t * a^i) % p = {left}")
            print(f"             (h[i+m] - h[i]) % p = {right}")

        if left == right:
            if verbose:
                print("  hash match ✅ -> verify characters to avoid collision")
            if s[i:i + m] == t:
                hits.append(i)
                if verbose:
                    print("  verified MATCH ✅")
            else:
                if verbose:
                    print("  collision ❌ (hash match but strings differ)")
        else:
            if verbose:
                print("  hash mismatch -> definitely not a match")

    if verbose:
        print("\nMatches at indices:", hits)

    return hits


# Example:
a = 911382323
p = 1_000_000_007
rabin_karp_trace("ababacababa", t="aba", a=a, p=p, verbose=True)

------------------------------------------------------------------------------------------
Rabin–Karp trace
s='ababacababa'
t='aba'
base a=911382323, mod p=1000000007
hash(t) (h_t) = 685316838
------------------------------------------------------------------------------------------

shift i=0
s: ababacababa
t: aba
compare hashes: (h_t * a^i) % p = 685316838
             (h[i+m] - h[i]) % p = 685316838
  hash match ✅ -> verify characters to avoid collision
  verified MATCH ✅

shift i=1
s: ababacababa
t:  aba
compare hashes: (h_t * a^i) % p = 435355145
             (h[i+m] - h[i]) % p = 283053689
  hash mismatch -> definitely not a match

shift i=2
s: ababacababa
t:   aba
compare hashes: (h_t * a^i) % p = 602676975
             (h[i+m] - h[i]) % p = 602676975
  hash match ✅ -> verify characters to avoid collision
  verified MATCH ✅

shift i=3
s: ababacababa
t:    aba
compare hashes: (h_t * a^i) % p = 649228966
             (h[i+m] - h[i]) % p = 436878875
  hash mismatch -> definitely no

[0, 2, 6, 8]

## **Final summary:**

### **Hashing**
- Expected collisions for random function: **E = N/M**
- Universal family guarantees: **P(collision) ≤ 1/M**
- Polynomial hash supports fast substring hashing.

### **Rabin–Karp**
- Rolling hash compares substring hashes in O(1)
- Total expected: **O(|s| + |t| + verification work)**

### **Hash tables**
| Method | Collision handling | Expected op time (α=N/M) | Notes |
|---|---|---:|---|
| Separate chaining | linked list per bucket | **O(α+1)** | easy deletes |
| Open addressing | probing in array | depends strongly on α | keep α small; Python uses open addressing + double hashing |